# Emerging Tech - Quantum Computing Notebook

## Problem 1 - Generating random boolean Funtions

In [7]:
# Random selections.
import random

# Numerical arrays.
import numpy as np

# Permutations and combinations.
import itertools as it

def random_constant_balanced():
    # All possible combinations of 4 boolean inputs
    # e.g. (False, False, False, False), (False, False, False, True), ...
    all_inputs = list(it.product([False, True], repeat=4))  # 16 combinations
    
    # Randomly pick constant or balanced
    function_type = random.choice(["constant", "balanced"])
    
    if function_type == "constant":
        # Either always return True or always return False
        constant_value = random.choice([True, False])
        def f(*args):
            return constant_value
    
    else:  # balanced
        # Pick exactly 8 of the 16 inputs to return True for
        true_inputs = set(map(tuple, random.sample(all_inputs, 8)))
        def f(*args):
            return tuple(args) in true_inputs
    
    return f

#### Test Case

In [8]:
f = random_constant_balanced()

# Try all 16 inputs and see what it returns
all_inputs = list(it.product([False, True], repeat=4))
results = [f(*inp) for inp in all_inputs]
print(results)
print("True count:", results.count(True))
print("False count:", results.count(False))

[False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False]
True count: 0
False count: 16


## Problem 2 - Classical Testing for Function Type

In [9]:
def determine_constant_balanced(f):
    # Generate all 16 possible combinations of 4 boolean inputs
    all_inputs = list(it.product([False, True], repeat=4))
    
    results = set()
    
    for inp in all_inputs:
        result = f(*inp)
        results.add(result)
        
        # If we've seen both True and False, it must be balanced
        if len(results) == 2:
            return "balanced"
    
    # Got through all inputs with only one unique result, must be constant
    return "constant"

### Test Case

In [10]:
# Run the test 5 times to see a mix of constant and balanced results
for i in range(5):
    
    # Generate a random constant or balanced function
    f = random_constant_balanced()
    
    print(determine_constant_balanced(f))

constant
constant
balanced
constant
constant


## Problem 3 - Quantum Oracles

In [11]:
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram

In [12]:

def oracle_constant_0():
    # f(x) = 0 always, so we do nothing to the circuit
    # y XOR 0 = y, output qubit is unchanged
    qc = QuantumCircuit(2)
    return qc

def oracle_constant_1():
    # f(x) = 1 always, so we flip the output qubit regardless of input
    # y XOR 1 = NOT y
    qc = QuantumCircuit(2)
    qc.x(1)  # X gate flips qubit 1 (the output qubit)
    return qc

def oracle_balanced_identity():
    # f(x) = x, so output depends on input
    # y XOR x, we use a CNOT gate controlled on input qubit
    qc = QuantumCircuit(2)
    qc.cx(0, 1)  # CNOT: flips qubit 1 if qubit 0 is 1
    return qc

def oracle_balanced_not():
    # f(x) = NOT x, so output is flipped version of input
    # flip input qubit first, then apply CNOT, then flip back
    qc = QuantumCircuit(2)
    qc.x(0)       # flip input qubit
    qc.cx(0, 1)   # CNOT
    qc.x(0)       # flip input qubit back
    return qc

### Test Oracle

In [13]:
def test_oracle(oracle, name):
    print(f"\nOracle: {name}")
    
    # Test with input 0
    qc = QuantumCircuit(2, 2)
    qc.compose(oracle, inplace=True)
    qc.measure([0, 1], [0, 1])
    
    simulator = Aer.get_backend('qasm_simulator')
    job = simulator.run(qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    print(f"  Input |00> gives: {counts}")
    
    # Test with input 1
    qc = QuantumCircuit(2, 2)
    qc.x(0)  # set input qubit to 1
    qc.compose(oracle, inplace=True)
    qc.measure([0, 1], [0, 1])
    
    job = simulator.run(qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    print(f"  Input |10> gives: {counts}")

# Test all four oracles
test_oracle(oracle_constant_0(), "Constant 0")
test_oracle(oracle_constant_1(), "Constant 1")
test_oracle(oracle_balanced_identity(), "Balanced Identity")
test_oracle(oracle_balanced_not(), "Balanced NOT")


Oracle: Constant 0
  Input |00> gives: {'00': 1}
  Input |10> gives: {'01': 1}

Oracle: Constant 1
  Input |00> gives: {'10': 1}
  Input |10> gives: {'11': 1}

Oracle: Balanced Identity
  Input |00> gives: {'00': 1}
  Input |10> gives: {'11': 1}

Oracle: Balanced NOT
  Input |00> gives: {'10': 1}
  Input |10> gives: {'01': 1}


## Problem 4 - Deutsch's Algorithm with Qiskit

In [14]:
from qiskit import QuantumCircuit
from qiskit_aer import Aer

In [15]:
def deutsch_algorithm(oracle):
    # Create circuit with 2 qubits and 1 classical bit for measurement
    qc = QuantumCircuit(2, 1)
    
    # Step 1: Set output qubit (qubit 1) to |1>
    qc.x(1)
    
    # Step 2: Apply Hadamard to both qubits to create superposition
    qc.h(0)
    qc.h(1)
    
    # Step 3: Apply the oracle (single query to the function)
    qc.compose(oracle, inplace=True)
    
    # Step 4: Apply Hadamard to input qubit to cause interference
    qc.h(0)
    
    # Step 5: Measure only the input qubit
    qc.measure(0, 0)
    
    return qc


In [16]:
def run_deutsch(oracle, oracle_name):
    # Build the circuit with this oracle
    qc = deutsch_algorithm(oracle)
    
    # Run on the simulator
    simulator = Aer.get_backend('qasm_simulator')
    job = simulator.run(qc, shots=1024)
    result = job.result()
    counts = result.get_counts()
    
    # If input qubit measured as 0 -> constant, if 1 -> balanced
    if '0' in counts and counts['0'] == 1024:
        conclusion = "constant"
    else:
        conclusion = "balanced"
    
    print(f"Oracle: {oracle_name}")
    print(f"Measurement results: {counts}")
    print(f"Conclusion: {conclusion}")
    print()

In [17]:
# Test with all four oracles from Problem 3
run_deutsch(oracle_constant_0(), "Constant 0")
run_deutsch(oracle_constant_1(), "Constant 1")
run_deutsch(oracle_balanced_identity(), "Balanced Identity")
run_deutsch(oracle_balanced_not(), "Balanced NOT")

Oracle: Constant 0
Measurement results: {'0': 1024}
Conclusion: constant

Oracle: Constant 1
Measurement results: {'0': 1024}
Conclusion: constant

Oracle: Balanced Identity
Measurement results: {'1': 1024}
Conclusion: balanced

Oracle: Balanced NOT
Measurement results: {'1': 1024}
Conclusion: balanced



In [18]:
# Draw the circuit for one example so you can see the structure
qc = deutsch_algorithm(oracle_balanced_identity())
print(qc.draw())

     ┌───┐          ┌───┐┌─┐
q_0: ┤ H ├───────■──┤ H ├┤M├
     ├───┤┌───┐┌─┴─┐└───┘└╥┘
q_1: ┤ X ├┤ H ├┤ X ├──────╫─
     └───┘└───┘└───┘      ║ 
c: 1/═════════════════════╩═
                          0 


## Problem 5 - Scaling to the Deutsch–Jozsa Algorithm